<a href="https://colab.research.google.com/github/sakshiR2001/sakshiR2001/blob/main/HR_POLICY_NAVIGATOR_IITKPG_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install requests
from google.colab import files

uploaded = files.upload()

CERT_PATH = "/content/vm05-ca.crt"

import os

if not os.path.exists(CERT_PATH):
    raise FileNotFoundError("Please upload vm05-ca.crt")

print("Certificate uploaded successfully!")
print("Certificate path:", CERT_PATH)

In [ ]:
import getpass

# Enter your API key securely
API_KEY = getpass.getpass("Enter your IIT KGP API key: ")

BASE_URL = "https://103.42.50.41:1033/v1"

MODEL = "Qwen/Qwen3-8B"

print("API configuration completed!")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

response = requests.get(
    f"{BASE_URL}/models",
    headers=headers,
    verify=CERT_PATH,
    timeout=30
)

print("Status:", response.status_code)

if response.status_code == 200:
    print("API connection successful!")
    print(response.json())
else:
    print(response.text)

In [ ]:
import os

PROJECT = "/content/HR-Policy-Navigator"

folders = [
    "data/raw",
    "data/processed",
    "data/evaluation",
    "notebooks",
    "src/ingestion",
    "src/retrieval",
    "src/llm",
    "src/evaluation"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT, folder), exist_ok=True)

print("Project folders created successfully!")

In [ ]:
files = {
    "README.md": """# HR Policy Navigator

A Generative AI and Agentic AI based HR Policy Navigator.

## Model
Qwen/Qwen3-8B through the assigned upGrad API.

## Project modules
- Document ingestion
- HR policy retrieval
- LLM-based question answering
- Evaluation framework
- Agentic workflows
""",

    "requirements.txt": """openai
httpx
pandas
numpy
pypdf
python-docx
sentence-transformers
faiss-cpu
streamlit
tqdm
""",

    ".gitignore": """.env
*.key
*.crt
__pycache__/
.ipynb_checkpoints/
data/raw/*
data/processed/*
!data/raw/.gitkeep
!data/processed/.gitkeep
""",

    ".env.example": """UPGRAD_API_KEY=your_api_key_here
UPGRAD_BASE_URL=https://103.42.50.41:1033/v1
UPGRAD_MODEL=Qwen/Qwen3-8B
""",

    "src/llm/qwen_client.py": """import os
import httpx
from openai import OpenAI

def get_client(ca_cert_path):
    return OpenAI(
        api_key=os.environ["UPGRAD_API_KEY"],
        base_url="https://103.42.50.41:1033/v1",
        http_client=httpx.Client(
            verify=ca_cert_path,
            timeout=120.0
        )
    )
""",

    "src/app.py": """def main():
    print("HR Policy Navigator - Initial Project Skeleton")

if __name__ == "__main__":
    main()
"""
}

for filename, content in files.items():
    path = os.path.join(PROJECT, filename)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

for folder in ["data/raw", "data/processed"]:
    open(os.path.join(PROJECT, folder, ".gitkeep"), "w").close()

print("Initial project files created!")

In [ ]:
import getpass
import os
import subprocess

GITHUB_USERNAME = input("Enter your GitHub username: ")
GITHUB_TOKEN = getpass.getpass("Enter your GitHub token: ")

REPO_URL = (
    "https://github.com/"
    "capstone-project-iitkgp/HR-Policy-Navigator.git"
)

def run(command):
    subprocess.run(
        command,
        cwd=PROJECT,
        check=True
    )

run(["git", "init"])
run(["git", "branch", "-M", "main"])
run(["git", "config", "user.name", GITHUB_USERNAME])
run(["git", "config", "user.email",
     input("Enter your GitHub email: ")])

run(["git", "add", "."])
run(["git", "commit", "-m", "Initial HR Policy Navigator project"])

# Keep the token out of the remote URL.
askpass = "/content/github-askpass.sh"

with open(askpass, "w") as f:
    f.write("""#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USERNAME" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
  *) exit 1 ;;
esac
""")

os.chmod(askpass, 0o700)

env = os.environ.copy()
env["GITHUB_USERNAME"] = GITHUB_USERNAME
env["GITHUB_TOKEN"] = GITHUB_TOKEN
env["GIT_ASKPASS"] = askpass
env["GIT_TERMINAL_PROMPT"] = "0"

subprocess.run(
    ["git", "remote", "add", "origin", REPO_URL],
    cwd=PROJECT,
    check=True
)

subprocess.run(
    ["git", "push", "-u", "origin", "main"],
    cwd=PROJECT,
    env=env,
    check=True
)

print("Initial project pushed to GitHub!")